### Silver to Gold: Building BI Ready Tables

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

catalog_name = "ecommerce"

### Combine category, brancd & products ...

In [ ]:
df_products = spark.table(f"{catalog_name}.silver.slv_products")
df_category = spark.table(f"{catalog_name}.silver.slv_categories")
df_brands = spark.table(f"{catalog_name}.silver.slv_brands")

In [ ]:
display(df_category.limit(2))
display(df_brands.limit(2))
display(df_products.limit(2))

In [ ]:
%%sql
USE ecommerce.gold;

CREATE OR REPLACE TABLE ecommerce.gold.dim_products AS

WITH CTE_category_brand AS (
    SELECT   
        b.category_code,
        c.category_name,
        brand_code,
        brand_name
    FROM ecommerce.silver.slv_brands b
    JOIN ecommerce.silver.slv_categories c
    ON b.category_code = c.category_code
)
SELECT 
    p.product_id,
    p.sku,
    p.category_code,
    COALESCE(cb.category_name, 'Not Available') AS category_name,
    p.brand_code,
    COALESCE(cb.brand_name, 'Not Available')   AS brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.source_file AS file_name,
    p.created_at AS ingest_timestamp
FROM ecommerce.silver.slv_products p 
LEFT JOIN CTE_category_brand cb 
ON p.brand_code = cb.brand_code

In [ ]:
%%sql
SELECT * FROM ecommerce.gold.dim_products LIMIT 5;

### Customer table ..

In [ ]:
df_customers = spark.table(f"{catalog_name}.silver.slv_customers")
display(df_customers.limit(2))

In [ ]:
# Add customer region..

# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  

country_state_map

In [ ]:
regiondata = []

for country, states in country_state_map.items():
    for state, region in states.items():
        regiondata.append((country, state, region))

regiondata

# Crate a region data frame..
df_region = spark.createDataFrame(regiondata, schema=["country", "state", "region"])

display(df_region.limit(10))

In [ ]:
gold_customers = df_customers.join(df_region, on=["country", "state"], how="left")

# set default value for region
gold_customers.fillna({"region": "Other"})
display(gold_customers.select(['customer_id', 'phone', 'country_code', 'country', "state", 'region', 'created_at', 'source_file']).limit(10))

In [ ]:
gold_customers.select(['customer_id', 'phone', 'country_code', 'country', "state", 'region', 'created_at', 'source_file'])\
    .write\
    .format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", True)\
    .saveAsTable(f"{catalog_name}.gold.dim_customers")

In [ ]:
%%sql
SELECT * FROM ecommerce.gold.dim_customers LIMIT 10;

### Calender ..

In [ ]:
df_calender = spark.table(f"{catalog_name}.silver.slv_calender")

display(df_calender.limit(10))

In [ ]:
#  Add date id
gold_calender = df_calender.withColumn("date_id", F.date_format("date", "yyyyMMdd").cast("int"))

# Add month name...
gold_calender = gold_calender.withColumn("month", F.date_format("date", "MMMM"))

# Add weekend..
gold_calender = gold_calender.withColumn("is_weekend", F.when(F.col("day_name").isin(["Saturday", "Sunday"]), 1).otherwise(0))

display(gold_calender.limit(10))

In [ ]:
gold_calender.select(['date_id', "date", 'year', 'month', 'day_name', 'quarter', 'week_of_year', 'is_weekend', 'created_at', 'source_file'])\
    .write\
    .format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", True)\
    .saveAsTable(f"{catalog_name}.gold.dim_calender")

In [ ]:
%%sql
SELECT * FROM ecommerce.gold.dim_calender LIMIT 10;